# Formula 1 — Comprehensive Statistical Portfolio (1950–2025)

> This report covers 10 analytical domains, 40+ interactive visualizations, and over 100 distinct metrics. Sections are self-contained and ordered from broad records to fine-grained performance indicators.

**Sections:**
1. Setup & Data Overview
2. Driver Records — Volume
3. Driver Efficiency — Era-Adjusted Ratios
4. Driver Racecraft — Positions Gained & Consistency
5. Prestige Achievements — Hat Tricks, Grand Slams, Pole Conversion
6. Constructor Dominance & Reliability
7. Circuit DNA — Attrition, Strategy & Pole Importance
8. Historical Trends — Reliability, Parity & Expansion
9. Nationality & Geography
10. Curiosities & Edge Cases


## 0. Setup & Data Overview

In [1]:

import warnings
warnings.filterwarnings('ignore')

import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# -- Resolve the project root robustly regardless of kernel launch directory --
# Walk up until we find a directory containing 'data/processed'
_cwd = Path().resolve()
_project_root = _cwd
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'data' / 'processed').exists():
        _project_root = _candidate
        break

PROCESSED_DATA_DIR = _project_root / 'data' / 'processed'

# ---- Visual constants ----
TEMPLATE = "plotly_dark"
C_RED    = "#E10600"
C_GOLD   = "#FFD700"
C_SILVER = "#C0C0C0"
C_BLUE   = "#4FC3F7"

px.defaults.template = TEMPLATE

# ---- Load data ----
df = pd.read_parquet(PROCESSED_DATA_DIR / "results.parquet")
df_drivers = pd.read_parquet(PROCESSED_DATA_DIR / "drivers.parquet")

# ---- Key derived columns ----
# Did Not Finish detection: status not in completion signals
FINISH_STATUSES = {'Finished', '+1 Lap', '+2 Laps', '+3 Laps', '+4 Laps',
                   '+5 Laps', '+6 Laps', '+7 Laps', '+8 Laps', '+9 Laps', 'Lapped'}
df['finished']        = df['status'].isin(FINISH_STATUSES)
df['is_dnf']          = ~df['finished'] & df['positionText'].isin(['R', 'D', 'E', 'W', 'F', 'N'])
df['on_podium']       = df['position'] <= 3
df['is_win']          = df['position'] == 1
df['is_pole']         = df['grid'] == 1
df['in_points']       = df['position'] <= 10   # modern top-10 scoring
df['fastest_lap']     = pd.to_numeric(df['fastest_lap_rank'], errors='coerce') == 1
df['decade']          = (df['season'] // 10) * 10
df['positions_gained']= df['grid'] - df['position']   # positive = gained

# Merge date-of-birth for age calculations
df = df.merge(df_drivers[['driverId', 'dateOfBirth']], on='driverId', how='left')
df['date'] = pd.to_datetime(df['date'])
df['dateOfBirth'] = pd.to_datetime(df['dateOfBirth'])
df['driver_age_at_race'] = (df['date'] - df['dateOfBirth']).dt.days / 365.25

print(f"Records loaded  : {len(df):,}")
print(f"Seasons covered : {df['season'].min()} – {df['season'].max()}")
print(f"Unique drivers  : {df['driver_fullname'].nunique()}")
print(f"Unique events   : {df['raceName'].nunique()}")
print(f"Unique teams    : {df['constructor_name'].nunique()}")


Records loaded  : 25,873
Seasons covered : 1950 – 2025
Unique drivers  : 817
Unique events   : 54
Unique teams    : 203


## 1. Driver Records — Volume

*Raw career totals: the absolute figures every fan knows.*

In [2]:

# ---- Aggregate per driver ----
drv = df.groupby('driver_fullname').agg(
    entries      = ('position', 'count'),
    wins         = ('is_win', 'sum'),
    podiums      = ('on_podium', 'sum'),
    poles        = ('is_pole', 'sum'),
    fastest_laps = ('fastest_lap', 'sum'),
    total_points = ('points', 'sum'),
    seasons      = ('season', 'nunique'),
    dnfs         = ('is_dnf', 'sum'),
    top10s       = ('in_points', 'sum'),
).reset_index().rename(columns={'driver_fullname': 'driver'})

drv = drv.sort_values('wins', ascending=False)
TOP_N = 25

# Chart 1a — Career Wins (Top 25)
fig = px.bar(
    drv.head(TOP_N).sort_values('wins'),
    x='wins', y='driver', orientation='h',
    color='wins', color_continuous_scale='Reds',
    title=f'Career Grand Prix Victories — Top {TOP_N}',
    labels={'wins': 'Victories', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 1b — Career Podiums
fig = px.bar(
    drv.sort_values('podiums', ascending=False).head(TOP_N).sort_values('podiums'),
    x='podiums', y='driver', orientation='h',
    color='podiums', color_continuous_scale='RdYlGn',
    title=f'Career Podium Finishes — Top {TOP_N}',
    labels={'podiums': 'Podiums', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 1c — Career Points
fig = px.bar(
    drv.sort_values('total_points', ascending=False).head(TOP_N).sort_values('total_points'),
    x='total_points', y='driver', orientation='h',
    color='total_points', color_continuous_scale='Blues',
    title=f'Career Championship Points — Top {TOP_N}',
    labels={'total_points': 'Points', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 1d — Most GP Entries (Longevity)
fig = px.bar(
    drv.sort_values('entries', ascending=False).head(TOP_N).sort_values('entries'),
    x='entries', y='driver', orientation='h',
    color='entries', color_continuous_scale='Purples',
    title=f'Career Grand Prix Entries — Top {TOP_N}',
    labels={'entries': 'GP Starts', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Table — Combined Top 15
drv.head(15)[['driver','entries','wins','podiums','poles','fastest_laps','total_points','seasons']]


,driver,entries,wins,podiums,poles,fastest_laps,total_points,seasons
491,Lewis Hamilton,380,105,202,104,68,4955.50,19
544,Michael Schumacher,308,91,155,68,21,1566.00,19
538,Max Verstappen,233,71,127,48,36,3301.50,11
726,Sebastian Vettel,300,53,122,57,38,3098.00,16
8,Alain Prost,202,51,106,33,0,798.50,13
57,Ayrton Senna,161,41,80,65,0,614.00,11
234,Fernando Alonso,428,32,106,22,25,2380.00,22
581,Nigel Mansell,190,31,59,32,0,482.00,15
358,Jackie Stewart,100,27,43,17,0,360.00,9
386,Jim Clark,73,25,32,34,0,274.00,9


## 2. Driver Efficiency — Era-Adjusted Ratios

*Raw totals favour modern era drivers with more races. Ratios level the playing field.*

*Minimum 20 GP starts applied to remove statistical outliers.*

In [3]:

# Efficiency metrics (min 20 entries)
eff = drv[drv['entries'] >= 20].copy()
eff['win_pct']     = (eff['wins']    / eff['entries'] * 100).round(2)
eff['podium_pct']  = (eff['podiums'] / eff['entries'] * 100).round(2)
eff['pole_pct']    = (eff['poles']   / eff['entries'] * 100).round(2)
eff['pts_per_gp']  = (eff['total_points'] / eff['entries']).round(2)
eff['dnf_pct']     = (eff['dnfs']    / eff['entries'] * 100).round(2)
eff['top10_pct']   = (eff['top10s']  / eff['entries'] * 100).round(2)

# Chart 2a — Win Percentage (min 20 entries)
top_win_pct = eff.sort_values('win_pct', ascending=False).head(20)
fig = px.bar(
    top_win_pct.sort_values('win_pct'),
    x='win_pct', y='driver', orientation='h',
    color='win_pct', color_continuous_scale='Reds',
    title='Win Percentage per Career Start (Min. 20 Entries)',
    labels={'win_pct': 'Win %', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 2b — Scatter: Win% vs Entries (bubble = total wins)
fig = px.scatter(
    eff.sort_values('wins', ascending=False).head(50),
    x='entries', y='win_pct',
    size='wins', size_max=50,
    hover_name='driver',
    color='win_pct', color_continuous_scale='Reds',
    title='Win Efficiency vs Career Longevity',
    labels={'entries': 'GP Entries', 'win_pct': 'Win %'}
)
fig.update_coloraxes(showscale=False)
fig.show()

# Chart 2c — Podium percentage
top_pod_pct = eff.sort_values('podium_pct', ascending=False).head(20)
fig = px.bar(
    top_pod_pct.sort_values('podium_pct'),
    x='podium_pct', y='driver', orientation='h',
    color='podium_pct', color_continuous_scale='RdYlGn',
    title='Podium Percentage per Career Start (Min. 20 Entries)',
    labels={'podium_pct': 'Podium %', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 2d — Points per GP start
top_pts = eff.sort_values('pts_per_gp', ascending=False).head(20)
fig = px.bar(
    top_pts.sort_values('pts_per_gp'),
    x='pts_per_gp', y='driver', orientation='h',
    color='pts_per_gp', color_continuous_scale='Blues',
    title='Average Points per GP Start (Min. 20 Entries)',
    labels={'pts_per_gp': 'Pts/GP', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

eff.sort_values('win_pct', ascending=False).head(15)[
    ['driver','entries','win_pct','podium_pct','pole_pct','pts_per_gp','dnf_pct']
]


,driver,entries,win_pct,podium_pct,pole_pct,pts_per_gp,dnf_pct
444,Juan Fangio,58,41.38,60.34,50.00,4.81,24.14
14,Alberto Ascari,36,36.11,47.22,38.89,3.89,38.89
386,Jim Clark,73,34.25,43.84,46.58,3.75,31.51
538,Max Verstappen,233,30.47,54.51,20.60,14.17,13.30
544,Michael Schumacher,308,29.55,50.32,22.08,5.08,21.75
491,Lewis Hamilton,380,27.63,53.16,27.37,13.04,8.42
358,Jackie Stewart,100,27.00,43.00,17.00,3.60,37.00
57,Ayrton Senna,161,25.47,49.69,40.37,3.81,32.92
8,Alain Prost,202,25.25,52.48,16.34,3.95,29.21
739,Stirling Moss,73,21.92,32.88,23.29,2.56,49.32


## 3. Driver Racecraft — Positions Gained & Consistency

*Who makes up ground on race day? Who protects their grid slot?*

In [4]:

# Only races where grid is valid (not 0 = pit lane start)
df_race = df[(df['grid'] > 0) & (df['position'] > 0)].copy()

racecraft = df_race.groupby('driver_fullname').agg(
    entries         = ('positions_gained', 'count'),
    avg_gained      = ('positions_gained', 'mean'),
    total_gained    = ('positions_gained', 'sum'),
    avg_grid        = ('grid', 'mean'),
    avg_finish      = ('position', 'mean'),
).reset_index().rename(columns={'driver_fullname': 'driver'})

racecraft['avg_gained'] = racecraft['avg_gained'].round(2)
racecraft['avg_grid']   = racecraft['avg_grid'].round(2)
racecraft['avg_finish'] = racecraft['avg_finish'].round(2)

# Min 50 races for reliability
rc_filtered = racecraft[racecraft['entries'] >= 50]

# Chart 3a — Top climbers (best avg positions gained)
climbers = rc_filtered.sort_values('avg_gained', ascending=False).head(20)
fig = px.bar(
    climbers.sort_values('avg_gained'),
    x='avg_gained', y='driver', orientation='h',
    color='avg_gained', color_continuous_scale='Greens',
    title='Average Positions Gained per Race (Min. 50 Starts)',
    labels={'avg_gained': 'Avg Pos. Gained', 'driver': ''}
)
fig.add_vline(x=0, line_dash='dash', line_color='white', opacity=0.4)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 3b — Avg grid vs avg finish (dot below diagonal = improves)
top_rc = rc_filtered.sort_values('entries', ascending=False).head(40)
fig = px.scatter(
    top_rc, x='avg_grid', y='avg_finish',
    hover_name='driver', size='entries',
    color='avg_gained', color_continuous_scale='RdYlGn',
    title='Average Grid vs Average Finish Position (Top 40 by GP Entries)',
    labels={'avg_grid': 'Avg Grid Position', 'avg_finish': 'Avg Finish Position'}
)
fig.add_shape(type='line', x0=1, y0=1, x1=20, y1=20,
              line=dict(color='white', dash='dash', width=1))
fig.add_annotation(x=15, y=14, text='Grid = Finish', showarrow=False,
                   font=dict(color='white', size=10))
fig.update_coloraxes(colorbar_title='Avg Gained')
fig.show()

rc_filtered.sort_values('avg_gained', ascending=False).head(15)[
    ['driver','entries','avg_grid','avg_finish','avg_gained','total_gained']
]


,driver,entries,avg_grid,avg_finish,avg_gained,total_gained
426,Jonathan Palmer,84,20.62,14.48,6.14,516
485,Luca Badoer,52,20.17,14.33,5.85,304
501,Marc Surer,82,17.38,12.28,5.10,418
628,Philippe Streiff,54,18.20,13.72,4.48,242
687,Rolf Stommelen,54,17.37,13.11,4.26,230
790,Érik Comas,59,18.10,13.85,4.25,251
22,Alex Caffi,56,18.84,14.80,4.04,226
630,Piercarlo Ghinzani,76,21.96,18.01,3.95,300
315,Henri Pescarolo,56,17.14,13.57,3.57,200
535,Mika Salo,110,14.75,12.00,2.75,302


## 4. Prestige Achievements

- **Hat Trick**: Pole position + Fastest Lap + Win in the same race
- **Grand Slam**: Pole + Win + Fastest Lap + Led every lap
  *(Approximated here as Pole + Win + Fastest Lap, since lap-level leadership is not in this dataset)*
- **Pole Conversion Rate**: % of poles that resulted in a race win

In [5]:

# Hat tricks = pole + win + fastest lap in SAME race
hat_tricks = df[(df['is_pole']) & (df['is_win']) & (df['fastest_lap'])].groupby(
    'driver_fullname').size().reset_index(name='hat_tricks')

# Pole Conversion Rate
poles_df   = df[df['is_pole']].groupby('driver_fullname').size().reset_index(name='pole_starts')
pole_wins  = df[(df['is_pole']) & (df['is_win'])].groupby('driver_fullname').size().reset_index(name='pole_wins')
pole_conv  = poles_df.merge(pole_wins, on='driver_fullname', how='left').fillna(0)
pole_conv['pole_conv_pct'] = (pole_conv['pole_wins'] / pole_conv['pole_starts'] * 100).round(2)
pole_conv = pole_conv[pole_conv['pole_starts'] >= 3].sort_values('pole_conv_pct', ascending=False)

# Chart 4a — Hat Tricks
fig = px.bar(
    hat_tricks.sort_values('hat_tricks', ascending=False).head(15).sort_values('hat_tricks'),
    x='hat_tricks', y='driver_fullname', orientation='h',
    color='hat_tricks', color_continuous_scale='Oranges',
    title='Hat Tricks: Pole + Fastest Lap + Win — Career Total',
    labels={'hat_tricks': 'Hat Tricks', 'driver_fullname': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()

# Chart 4b — Pole Conversion Rate (min 3 poles)
fig = px.bar(
    pole_conv.head(20).sort_values('pole_conv_pct'),
    x='pole_conv_pct', y='driver_fullname', orientation='h',
    color='pole_conv_pct', color_continuous_scale='Reds',
    title='Pole Position Conversion Rate (Min. 3 Pole Positions)',
    labels={'pole_conv_pct': 'Pole to Win %', 'driver_fullname': ''},
    hover_data=['pole_starts', 'pole_wins']
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

print("=== Hat Trick Leaders ===")
print(hat_tricks.sort_values('hat_tricks', ascending=False).head(10).to_string(index=False))
print()
print("=== Pole Conversion Leaders ===")
print(pole_conv.head(15)[['driver_fullname','pole_starts','pole_wins','pole_conv_pct']].to_string(index=False))


=== Hat Trick Leaders ===
   driver_fullname  hat_tricks
    Lewis Hamilton          19
    Max Verstappen          14
  Sebastian Vettel           8
Michael Schumacher           7
   Fernando Alonso           5
      Felipe Massa           4
      Lando Norris           3
      Nico Rosberg           3
     Oscar Piastri           3
    George Russell           2

=== Pole Conversion Leaders ===
   driver_fullname  pole_starts  pole_wins  pole_conv_pct
    Max Verstappen           48       37.0          77.08
    Jody Scheckter            3        2.0          66.67
     Oscar Piastri            6        4.0          66.67
Emerson Fittipaldi            6        4.0          66.67
    Alberto Ascari           14        9.0          64.29
   Fernando Alonso           22       14.0          63.64
     Jenson Button            8        5.0          62.50
Michael Schumacher           68       40.0          58.82
    Lewis Hamilton          104       61.0          58.65
       Alain Prost  

## 5. Age Records

*Youngest and oldest to achieve key milestones.*

In [6]:

wins_df    = df[df['is_win'] & df['driver_age_at_race'].notna()].copy()
podiums_df = df[df['on_podium'] & df['driver_age_at_race'].notna()].copy()
poles_df_a = df[df['is_pole'] & df['driver_age_at_race'].notna()].copy()

# Youngest/Oldest winners
youngest_wins = wins_df.loc[wins_df.groupby('driver_fullname')['driver_age_at_race'].idxmin()]
youngest_wins = youngest_wins.sort_values('driver_age_at_race').head(10)[
    ['driver_fullname','season','raceName','driver_age_at_race']].rename(
    columns={'driver_age_at_race': 'age_years'})
youngest_wins['age_years'] = youngest_wins['age_years'].round(2)

oldest_wins = wins_df.loc[wins_df.groupby('driver_fullname')['driver_age_at_race'].idxmax()]
oldest_wins = oldest_wins.sort_values('driver_age_at_race', ascending=False).head(10)[
    ['driver_fullname','season','raceName','driver_age_at_race']].rename(
    columns={'driver_age_at_race': 'age_years'})
oldest_wins['age_years'] = oldest_wins['age_years'].round(2)

# Chart 5a — Youngest winners
fig = go.Figure([go.Bar(
    x=youngest_wins['age_years'],
    y=youngest_wins['driver_fullname'] + ' (' + youngest_wins['season'].astype(str) + ')',
    orientation='h',
    marker_color=C_RED,
    text=youngest_wins['age_years'].astype(str) + ' yrs',
    textposition='outside'
)])
fig.update_layout(
    title='Youngest Race Winners in F1 History',
    template=TEMPLATE, xaxis_title='Age at First Win (years)',
    yaxis={'categoryorder': 'total ascending'}, height=500
)
fig.show()

# Chart 5b — Oldest winners
fig = go.Figure([go.Bar(
    x=oldest_wins['age_years'],
    y=oldest_wins['driver_fullname'] + ' (' + oldest_wins['season'].astype(str) + ')',
    orientation='h',
    marker_color=C_GOLD,
    text=oldest_wins['age_years'].astype(str) + ' yrs',
    textposition='outside'
)])
fig.update_layout(
    title='Oldest Race Winners in F1 History (Last Win)',
    template=TEMPLATE, xaxis_title='Age at Last Win (years)',
    yaxis={'categoryorder': 'total ascending'}, height=500
)
fig.show()

print("=== 10 Youngest Race Winners ===")
print(youngest_wins.to_string(index=False))
print()
print("=== 10 Oldest Race Winners (at Last Win) ===")
print(oldest_wins.to_string(index=False))


=== 10 Youngest Race Winners ===
 driver_fullname  season                 raceName  age_years
  Max Verstappen    2016       Spanish Grand Prix      18.62
Sebastian Vettel    2008       Italian Grand Prix      21.20
 Charles Leclerc    2019       Belgian Grand Prix      21.88
 Fernando Alonso    2003     Hungarian Grand Prix      22.07
    Troy Ruttman    1952         Indianapolis 500      22.22
   Bruce McLaren    1959 United States Grand Prix      22.28
  Lewis Hamilton    2007      Canadian Grand Prix      22.42
   Oscar Piastri    2024     Hungarian Grand Prix      23.29
  Kimi Räikkönen    2003     Malaysian Grand Prix      23.43
   Robert Kubica    2008      Canadian Grand Prix      23.50

=== 10 Oldest Race Winners (at Last Win) ===
    driver_fullname  season                 raceName  age_years
      Luigi Fagioli    1951        French Grand Prix      53.06
        Nino Farina    1953        German Grand Prix      46.76
        Juan Fangio    1957        German Grand Prix      

## 6. Constructor Dominance & Reliability

In [7]:

const = df.groupby('constructor_name').agg(
    entries      = ('position', 'count'),
    wins         = ('is_win', 'sum'),
    podiums      = ('on_podium', 'sum'),
    poles        = ('is_pole', 'sum'),
    fastest_laps = ('fastest_lap', 'sum'),
    total_points = ('total_points', 'sum') if 'total_points' in df.columns else ('points', 'sum'),
    dnfs         = ('is_dnf', 'sum'),
    seasons      = ('season', 'nunique'),
    drivers_used = ('driverId', 'nunique'),
).reset_index().rename(columns={'constructor_name': 'constructor'})

const = const.sort_values('wins', ascending=False)

# Efficiency
const['win_pct']    = (const['wins']   / const['entries'] * 100).round(2)
const['dnf_pct']    = (const['dnfs']   / const['entries'] * 100).round(2)
const['pts_per_gp'] = (const['total_points'] / const['entries']).round(2)

# Chart 6a — Constructor Wins
fig = px.bar(
    const.head(15).sort_values('wins'),
    x='wins', y='constructor', orientation='h',
    color='wins', color_continuous_scale='Reds',
    title='All-Time Constructor Grand Prix Victories — Top 15',
    labels={'wins': 'Victories', 'constructor': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 6b — Constructor Reliability (DNF %)
reliable = const[const['entries'] >= 50].sort_values('dnf_pct')
fig = px.bar(
    reliable.head(20).sort_values('dnf_pct', ascending=False),
    x='dnf_pct', y='constructor', orientation='h',
    color='dnf_pct', color_continuous_scale='RdYlGn_r',
    title='Constructor Reliability: DNF Rate (Min. 50 Entries)',
    labels={'dnf_pct': 'DNF %', 'constructor': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 6c — Constructor Win % (min 50 entries)
fig = px.bar(
    const[const['entries'] >= 50].sort_values('win_pct', ascending=False).head(15).sort_values('win_pct'),
    x='win_pct', y='constructor', orientation='h',
    color='win_pct', color_continuous_scale='Reds',
    title='Constructor Win Percentage (Min. 50 Entries)',
    labels={'win_pct': 'Win %', 'constructor': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()

# Chart 6d — Constructor points by season (Top 8 all-time teams)
top8_const = const.head(8)['constructor'].tolist()
season_pts = df[df['constructor_name'].isin(top8_const)].groupby(
    ['season', 'constructor_name'])['points'].sum().reset_index()
fig = px.line(season_pts, x='season', y='points', color='constructor_name',
              title='Constructor Points per Season — Top 8 Teams',
              labels={'points': 'Season Points', 'season': 'Year', 'constructor_name': 'Constructor'})
fig.update_layout(height=500)
fig.show()

const.head(15)[['constructor','entries','wins','win_pct','podiums','poles','dnf_pct','seasons','drivers_used']]


,constructor,entries,wins,win_pct,podiums,poles,dnf_pct,seasons,drivers_used
69,Ferrari,2480,249,10.04,848,259,27.26,76,98
126,McLaren,1951,199,10.20,542,177,26.50,56,53
131,Mercedes,699,131,18.74,310,144,10.59,18,13
158,Red Bull,836,130,15.55,297,111,15.55,21,14
200,Williams,1713,114,6.65,316,128,26.74,50,59
183,Team Lotus,831,45,5.42,114,61,48.01,29,57
159,Renault,785,35,4.46,103,51,32.74,24,26
22,Benetton,519,27,5.20,102,15,35.26,16,17
194,Tyrrell,850,23,2.71,77,14,46.24,29,46
24,Brabham,616,23,3.73,78,26,51.46,22,43


## 7. Circuit DNA — Attrition, Strategy & Pole Importance

In [8]:

# Circuit-level attrition
circ = df.groupby(['circuitId', 'raceName']).agg(
    total_starters = ('position', 'count'),
    dnfs           = ('is_dnf', 'sum'),
    pole_wins      = ('is_pole', lambda x: (x & df.loc[x.index, 'is_win']).sum()),
    races_held     = ('round', 'nunique'),
).reset_index()
circ['attrition_pct'] = (circ['dnfs'] / circ['total_starters'] * 100).round(2)
circ['pole_win_pct']  = (circ['pole_wins'] / circ['races_held'] * 100).round(2)

# Chart 7a — Most punishing circuits
fig = px.bar(
    circ[circ['races_held'] >= 5].sort_values('attrition_pct', ascending=False).head(20),
    x='attrition_pct', y='raceName', orientation='h',
    color='attrition_pct', color_continuous_scale='Reds',
    title='Circuit Severity: Average DNF Rate (Min. 5 Editions)',
    labels={'attrition_pct': 'DNF %', 'raceName': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 7b — Pole-to-Win conversion by circuit (Monaco factor)
fig = px.bar(
    circ[circ['races_held'] >= 10].sort_values('pole_win_pct', ascending=False).head(20).sort_values('pole_win_pct'),
    x='pole_win_pct', y='raceName', orientation='h',
    color='pole_win_pct', color_continuous_scale='Oranges',
    title='Pole Position Win Conversion by Circuit (Min. 10 Editions)',
    labels={'pole_win_pct': 'Pole to Win %', 'raceName': ''},
    hover_data=['races_held']
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Most dominant driver per circuit
circuit_wins = df[df['is_win']].groupby(['raceName', 'driver_fullname']).size().reset_index(name='wins')
most_dominant = circuit_wins.loc[circuit_wins.groupby('raceName')['wins'].idxmax()].rename(
    columns={'driver_fullname': 'most_wins_driver'})
most_dominant = most_dominant[most_dominant['wins'] >= 2].sort_values('wins', ascending=False)

print("=== Most Dominant Drivers by Circuit ===")
print(most_dominant.head(20).to_string(index=False))


=== Most Dominant Drivers by Circuit ===
                raceName   most_wins_driver  wins
      British Grand Prix     Lewis Hamilton     9
       French Grand Prix Michael Schumacher     8
    Hungarian Grand Prix     Lewis Hamilton     8
     Canadian Grand Prix     Lewis Hamilton     7
   San Marino Grand Prix Michael Schumacher     7
      Chinese Grand Prix     Lewis Hamilton     6
      Spanish Grand Prix     Lewis Hamilton     6
     European Grand Prix Michael Schumacher     6
    Brazilian Grand Prix        Alain Prost     6
United States Grand Prix     Lewis Hamilton     6
      Belgian Grand Prix Michael Schumacher     6
       Monaco Grand Prix       Ayrton Senna     6
     Japanese Grand Prix Michael Schumacher     6
      Italian Grand Prix     Lewis Hamilton     5
      Russian Grand Prix     Lewis Hamilton     5
      Bahrain Grand Prix     Lewis Hamilton     5
    Abu Dhabi Grand Prix     Lewis Hamilton     5
    Singapore Grand Prix   Sebastian Vettel     5
   Austra

## 8. Historical Trends — Reliability, Parity & Calendar Expansion

In [9]:

# Chart 8a — DNF rate by decade
dnf_decade = df.groupby('decade').agg(
    total   = ('is_dnf', 'count'),
    dnfs    = ('is_dnf', 'sum')
).reset_index()
dnf_decade['dnf_pct'] = (dnf_decade['dnfs'] / dnf_decade['total'] * 100).round(2)

fig = px.bar(dnf_decade, x='decade', y='dnf_pct',
             color='dnf_pct', color_continuous_scale='RdYlGn_r',
             title='DNF Rate Evolution by Decade (Technical Reliability)',
             labels={'dnf_pct': 'DNF %', 'decade': 'Decade'})
fig.update_coloraxes(showscale=False)
fig.show()

# Chart 8b — Races per season
gps_per_season = df.groupby('season')['round'].max().reset_index(name='num_gps')
fig = px.area(gps_per_season, x='season', y='num_gps',
              title='Formula 1 Calendar Expansion: Grand Prix per Season',
              labels={'num_gps': 'Number of Races', 'season': 'Year'})
fig.update_traces(line_color=C_RED, fillcolor='rgba(225,6,0,0.2)')
fig.show()

# Chart 8c — Unique winners per season (competitiveness index)
unique_winners = df[df['is_win']].groupby('season')['driver_fullname'].nunique().reset_index(name='unique_winners')
fig = px.line(unique_winners, x='season', y='unique_winners', markers=True,
              title='Competitive Diversity: Unique Race Winners per Season',
              labels={'unique_winners': 'Distinct Winners', 'season': 'Year'})
fig.add_hline(y=unique_winners['unique_winners'].mean(), line_dash='dash',
              annotation_text=f"Average: {unique_winners['unique_winners'].mean():.1f}",
              annotation_position='top right')
fig.update_traces(line_color=C_GOLD)
fig.show()

# Chart 8d — Most dominant season (highest win % by a single driver)
season_dom = df.groupby(['season', 'driver_fullname']).agg(
    wins    = ('is_win', 'sum'),
    entries = ('position', 'count')
).reset_index()
season_dom['win_pct'] = season_dom['wins'] / season_dom['entries'] * 100
# Find the most dominant driver each season
best_season = season_dom.loc[season_dom.groupby('season')['win_pct'].idxmax()].sort_values('win_pct', ascending=False)

print("=== Most Dominant Single-Season Performances (Win %) ===")
print(best_season.head(15)[['season','driver_fullname','wins','entries','win_pct']].to_string(index=False))


=== Most Dominant Single-Season Performances (Win %) ===
 season    driver_fullname  wins  entries    win_pct
   1950    Johnnie Parsons     1        1 100.000000
   1951        Lee Wallard     1        1 100.000000
   1952       Troy Ruttman     1        1 100.000000
   1953      Bill Vukovich     1        1 100.000000
   1954      Bill Vukovich     1        1 100.000000
   1955       Bob Sweikert     1        1 100.000000
   1956       Pat Flaherty     1        1 100.000000
   1957          Sam Hanks     1        1 100.000000
   1958        Jimmy Bryan     1        1 100.000000
   1960       Jim Rathmann     1        1 100.000000
   1968          Jim Clark     1        1 100.000000
   2023     Max Verstappen    19       22  86.363636
   2004 Michael Schumacher    13       18  72.222222
   1963          Jim Clark     7       10  70.000000
   2020     Lewis Hamilton    11       16  68.750000


## 9. Nationality & Geography

In [10]:

# Nationality requires driver table nationality column
if 'nationality' in df_drivers.columns:
    nat_df = df.merge(df_drivers[['driverId','nationality']], on='driverId', how='left')
    nat_wins = nat_df[nat_df['is_win']].groupby('nationality').size().reset_index(name='wins')
    nat_entries = nat_df.groupby('nationality')['position'].count().reset_index(name='entries')
    nat_stats = nat_wins.merge(nat_entries, on='nationality')
    nat_stats['win_pct'] = (nat_stats['wins'] / nat_stats['entries'] * 100).round(2)
    nat_stats = nat_stats.sort_values('wins', ascending=False)

    # Chart 9a — Wins by nationality
    fig = px.bar(
        nat_stats.head(15).sort_values('wins'),
        x='wins', y='nationality', orientation='h',
        color='wins', color_continuous_scale='Blues',
        title='Grand Prix Victories by Driver Nationality — Top 15',
        labels={'wins': 'Victories', 'nationality': ''}
    )
    fig.update_coloraxes(showscale=False)
    fig.update_layout(height=550)
    fig.show()

    # Chart 9b — Pie chart of wins distribution
    fig = px.pie(nat_stats.head(10), values='wins', names='nationality',
                 title='Share of Grand Prix Victories by Nation (Top 10)',
                 color_discrete_sequence=px.colors.qualitative.Dark24)
    fig.show()
else:
    print("Nationality data not available in drivers table.")


## 10. Curiosities & Edge Cases

*Rare feats, unlucky records, and unusual statistical patterns.*

In [11]:

# --- 10a: Most podiums without a win ---
no_wins = drv[drv['wins'] == 0].sort_values('podiums', ascending=False).head(10)
print("=== Most Podiums Without a Single Victory ===")
print(no_wins[['driver','entries','podiums','total_points']].to_string(index=False))

# Chart 10a
fig = px.bar(
    no_wins.sort_values('podiums'),
    x='podiums', y='driver', orientation='h',
    color='podiums', color_continuous_scale='Oranges',
    title='Most Podiums Without a Race Win',
    labels={'podiums': 'Career Podiums', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.show()

# --- 10b: Most constructors driven for (most nomadic) ---
constructors_driven = df.groupby('driver_fullname')['constructor_name'].nunique().reset_index(
    name='num_constructors').sort_values('num_constructors', ascending=False)
print()
print("=== Most Constructors Driven For ===")
print(constructors_driven.head(10).to_string(index=False))

# --- 10c: Most GP starts before first win (perseverance) ---
first_win_round = df[df['is_win']].groupby('driver_fullname').agg(
    first_win_season = ('season', 'min')
).reset_index()
# Count entries before first win
races_before_win = []
for _, row in first_win_round.iterrows():
    driver = row['driver_fullname']
    first_win_yr = row['first_win_season']
    entries_before = df[(df['driver_fullname'] == driver) & (df['season'] <= first_win_yr)].shape[0]
    races_before_win.append({'driver': driver, 'first_win_year': first_win_yr, 'starts_to_first_win': entries_before})

rbw_df = pd.DataFrame(races_before_win).sort_values('starts_to_first_win', ascending=False).head(15)

print()
print("=== Most GP Starts Before First Victory ===")
print(rbw_df.to_string(index=False))

# Chart 10c
fig = px.bar(
    rbw_df.sort_values('starts_to_first_win'),
    x='starts_to_first_win', y='driver', orientation='h',
    color='starts_to_first_win', color_continuous_scale='Purples',
    title='GP Starts Before First Career Victory',
    labels={'starts_to_first_win': 'Starts Before First Win', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=550)
fig.show()

# --- 10d: Retirement cause breakdown ---
dnf_causes = df[df['is_dnf']].groupby('status').size().sort_values(ascending=False).head(20)
fig = px.bar(
    dnf_causes.reset_index().rename(columns={'index': 'cause', 'status': 'cause', 0: 'count'}),
    x=dnf_causes.values, y=dnf_causes.index, orientation='h',
    color=dnf_causes.values, color_continuous_scale='Reds',
    title='DNF Causes — Historical Breakdown',
    labels={'x': 'Occurrences', 'y': 'Retirement Cause'}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# --- 10e: Comeback wins (largest grid position that resulted in a win) ---
comeback_wins = df[df['is_win']].sort_values('grid', ascending=False).head(15)[
    ['driver_fullname', 'season', 'raceName', 'grid', 'position']
].rename(columns={'driver_fullname': 'driver', 'grid': 'start_position'})

print()
print("=== Greatest Comeback Victories (Started Furthest From Pole) ===")
print(comeback_wins.to_string(index=False))
fig = px.bar(
    comeback_wins.sort_values('start_position'),
    x='start_position',
    y=comeback_wins['driver'] + ' (' + comeback_wins['season'].astype(str) + ' ' + comeback_wins['raceName'] + ')',
    orientation='h',
    color='start_position', color_continuous_scale='Reds',
    title='Greatest Comeback Victories (Starting Position When Winning)',
    labels={'x': 'Grid Position', 'y': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()


=== Most Podiums Without a Single Victory ===
           driver  entries  podiums  total_points
    Nick Heidfeld      184       13        259.00
 Stefan Johansson       79       12         88.00
       Chris Amon      102       11         83.00
  Romain Grosjean      181       10        391.00
       Jean Behra       57        9         51.14
    Eddie Cheever      132        9         70.00
   Martin Brundle      158        9         98.00
  Luigi Villoresi       35        8         49.00
Andrea de Cesaris      209        5         59.00
    Derek Warwick      148        4         71.00



=== Most Constructors Driven For ===
    driver_fullname  num_constructors
Maurice Trintignant                13
         Chris Amon                13
      Stirling Moss                12
         Jo Bonnier                12
 Jean-Pierre Jarier                11
         Dan Gurney                10
  Andrea de Cesaris                10
         Jacky Ickx                10
       John Surtees                 9
      Roy Salvadori                 9



=== Most GP Starts Before First Victory ===
              driver  first_win_year  starts_to_first_win
        Sergio Pérez            2020                  193
        Carlos Sainz            2022                  163
         Mark Webber            2009                  140
  Rubens Barrichello            2000                  131
        Jarno Trulli            2004                  130
        Lando Norris            2024                  128
        Nico Rosberg            2012                  128
Giancarlo Fisichella            2003                  124
       Jenson Button            2006                  120
     Thierry Boutsen            1989                  105
          Jean Alesi            1995                  103
     Valtteri Bottas            2017                   98
       Mika Häkkinen            1997                   97
        Eddie Irvine            1999                   97
        Esteban Ocon            2021                   89



=== Greatest Comeback Victories (Started Furthest From Pole) ===
            driver  season                      raceName  start_position  position
       John Watson    1983 United States Grand Prix West              22         1
     Bill Vukovich    1954              Indianapolis 500              19         1
Rubens Barrichello    2000             German Grand Prix              18         1
    Max Verstappen    2024          São Paulo Grand Prix              17         1
       John Watson    1982            Detroit Grand Prix              17         1
    Kimi Räikkönen    2005           Japanese Grand Prix              17         1
Michael Schumacher    1995            Belgian Grand Prix              16         1
    Jackie Stewart    1973      South African Grand Prix              16         1
   Fernando Alonso    2008          Singapore Grand Prix              15         1
      Bob Sweikert    1955              Indianapolis 500              14         1
     Jenson Button   

## 11. Consecutive Streaks

*The longest uninterrupted runs of elite performance — the hardest records to match.*

In [12]:

# Sort chronologically
df_sorted = df.sort_values(['driver_fullname', 'date']).copy()

def longest_streak(series):
    # Compute the longest consecutive True run in a boolean series
    max_streak = streak = 0
    for val in series:
        if val:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak

# Win streaks
win_streaks = df_sorted.groupby('driver_fullname')['is_win'].apply(longest_streak).reset_index()
win_streaks.columns = ['driver', 'longest_win_streak']

# Podium streaks
pod_streaks = df_sorted.groupby('driver_fullname')['on_podium'].apply(longest_streak).reset_index()
pod_streaks.columns = ['driver', 'longest_podium_streak']

# Points streaks (top-10)
pts_streaks = df_sorted.groupby('driver_fullname')['in_points'].apply(longest_streak).reset_index()
pts_streaks.columns = ['driver', 'longest_points_streak']

# Merge
all_streaks = win_streaks.merge(pod_streaks, on='driver').merge(pts_streaks, on='driver')
all_streaks = all_streaks.merge(drv[['driver', 'entries']], on='driver')

# Chart 11a — Longest win streaks
fig = px.bar(
    all_streaks.sort_values('longest_win_streak', ascending=False).head(15).sort_values('longest_win_streak'),
    x='longest_win_streak', y='driver', orientation='h',
    color='longest_win_streak', color_continuous_scale='Reds',
    title='Longest Consecutive Win Streaks',
    labels={'longest_win_streak': 'Consecutive Wins', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()

# Chart 11b — Longest podium streaks
fig = px.bar(
    all_streaks.sort_values('longest_podium_streak', ascending=False).head(15).sort_values('longest_podium_streak'),
    x='longest_podium_streak', y='driver', orientation='h',
    color='longest_podium_streak', color_continuous_scale='RdYlGn',
    title='Longest Consecutive Podium Streaks',
    labels={'longest_podium_streak': 'Consecutive Podiums', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()

# Chart 11c — Longest points streaks (top-10 finishes)
fig = px.bar(
    all_streaks.sort_values('longest_points_streak', ascending=False).head(15).sort_values('longest_points_streak'),
    x='longest_points_streak', y='driver', orientation='h',
    color='longest_points_streak', color_continuous_scale='Blues',
    title='Longest Consecutive Points Finish Streaks (Top-10)',
    labels={'longest_points_streak': 'Consecutive Points Finishes', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()

all_streaks.sort_values('longest_win_streak', ascending=False).head(15)[
    ['driver', 'entries', 'longest_win_streak', 'longest_podium_streak', 'longest_points_streak']
]


,driver,entries,longest_win_streak,longest_podium_streak,longest_points_streak
538,Max Verstappen,233,10,15,43
726,Sebastian Vettel,300,9,11,21
14,Alberto Ascari,36,9,9,12
544,Michael Schumacher,308,7,19,24
578,Nico Rosberg,206,7,9,17
386,Jim Clark,73,6,9,13
581,Nigel Mansell,190,5,7,8
491,Lewis Hamilton,380,5,16,54
349,Jack Brabham,129,5,5,7
444,Juan Fangio,58,4,5,11


## 12. Teammate Head-to-Head

*Within the same team and same race, who beats their teammate most consistently? The purest measure of driver quality, removing the car variable.*

In [13]:

# Build teammate pairings per race
race_results = df[df['position'].notna() & (df['position'] > 0)].copy()
race_ids = ['season', 'round', 'constructor_name']

pairs = race_results.merge(
    race_results[race_ids + ['driver_fullname', 'position']],
    on=race_ids, suffixes=('_a', '_b')
)
pairs = pairs[pairs['driver_fullname_a'] < pairs['driver_fullname_b']].copy()
pairs['a_won'] = pairs['position_a'] < pairs['position_b']

htm = pairs.groupby(['driver_fullname_a', 'driver_fullname_b']).agg(
    races  = ('a_won', 'count'),
    a_wins = ('a_won', 'sum'),
).reset_index()
htm['b_wins'] = htm['races'] - htm['a_wins']
htm['a_win_pct'] = (htm['a_wins'] / htm['races'] * 100).round(1)
htm['gap_from_50'] = (htm['a_win_pct'] - 50).abs().round(1)
htm['label'] = htm['driver_fullname_a'] + '  vs  ' + htm['driver_fullname_b']
htm['dominant_driver'] = htm.apply(
    lambda r: r['driver_fullname_a'] if r['a_wins'] >= r['b_wins'] else r['driver_fullname_b'], axis=1
)

# ---- CHART: only truly contested duels ----
# Rule: min 10 shared races AND the dominant driver wins at most 65% (gap <= 15)
# This eliminates one-sided pairings like Verstappen/Tsunoda (95%), Verstappen/Gasly (92%)
contested = htm[(htm['races'] >= 10) & (htm['gap_from_50'] <= 15)].sort_values('gap_from_50')

print(f"Truly contested teammate rivalries (max 65/35 split, min 10 races): {len(contested)}")

fig = px.bar(
    contested.head(25).sort_values('gap_from_50', ascending=False),
    x='a_win_pct', y='label', orientation='h',
    color='gap_from_50', color_continuous_scale='RdYlGn_r',
    hover_data=['races', 'a_wins', 'b_wins', 'dominant_driver'],
    title='Most Closely Contested Teammate Rivalries<br><sup>Only duels with max 65/35 split — Min. 10 shared races</sup>',
    labels={'a_win_pct': 'Win % (Driver A alphabetical)', 'label': '', 'gap_from_50': 'Gap from 50/50'},
    range_color=[0, 15]
)
fig.add_vline(x=50, line_dash='dash', line_color='white',
              annotation_text='50 / 50', annotation_position='top',
              annotation_font_color='white')
fig.update_coloraxes(colorbar_title='Gap from 50%')
fig.update_layout(height=max(500, min(len(contested), 25) * 32))
fig.show()

# Table — full list of contested rivalries
contested[['label', 'races', 'a_wins', 'b_wins', 'a_win_pct', 'gap_from_50', 'dominant_driver']]


Truly contested teammate rivalries (max 65/35 split, min 10 races): 307


,label,races,a_wins,b_wins,a_win_pct,gap_from_50,dominant_driver
324,Andrea de Cesaris vs Bruno Giacomelli,18,9,9,50.0,0.0,Andrea de Cesaris
331,Andrea de Cesaris vs Mauro Baldi,14,7,7,50.0,0.0,Andrea de Cesaris
338,Andrea de Cesaris vs Ukyo Katayama,16,8,8,50.0,0.0,Andrea de Cesaris
555,Ayrton Senna vs Elio de Angelis,16,8,8,50.0,0.0,Ayrton Senna
1075,Bruce McLaren vs Maurice Trintignant,10,5,5,50.0,0.0,Bruce McLaren
...,...,...,...,...,...,...,...
2540,Giancarlo Fisichella vs Jenson Button,17,11,6,64.7,14.7,Giancarlo Fisichella
4441,Pedro Diniz vs Roberto Moreno,17,11,6,64.7,14.7,Pedro Diniz
4414,Patrick Depailler vs Ronnie Peterson,17,11,6,64.7,14.7,Patrick Depailler
2991,Jack Brabham vs Roy Salvadori,20,7,13,35.0,15.0,Roy Salvadori


## 13. Season Championship Battles

*How close were the title fights? Seasons decided by the narrowest margins.*

In [14]:

# Season-end standings: total points per driver per season
season_pts = df.groupby(['season', 'driver_fullname'])['points'].sum().reset_index()
season_pts = season_pts.sort_values(['season', 'points'], ascending=[True, False])

# Top-2 per season
top2 = season_pts.groupby('season').head(2).groupby('season').agg(
    champion = ('driver_fullname', 'first'),
    runner_up = ('driver_fullname', lambda x: x.iloc[1] if len(x) > 1 else 'N/A'),
    champion_pts = ('points', 'first'),
    runner_up_pts = ('points', lambda x: x.iloc[1] if len(x) > 1 else 0),
).reset_index()
top2['gap_pts'] = top2['champion_pts'] - top2['runner_up_pts']
top2 = top2.sort_values('gap_pts')

# Chart 13a — Narrowest title fights
fig = px.bar(
    top2.head(20).sort_values('gap_pts', ascending=False),
    x='gap_pts', y='season', orientation='h',
    color='gap_pts', color_continuous_scale='RdYlGn',
    hover_data=['champion', 'runner_up', 'champion_pts', 'runner_up_pts'],
    title='Narrowest Championship Battles (Smallest Points Gap, Top-20)',
    labels={'gap_pts': 'Points Gap (Champion - Runner-Up)', 'season': 'Season'},
    text='gap_pts'
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 13b — Most dominant seasons (largest gap)
fig = px.bar(
    top2.sort_values('gap_pts', ascending=False).head(20).sort_values('gap_pts'),
    x='gap_pts', y='season', orientation='h',
    color='gap_pts', color_continuous_scale='Reds',
    hover_data=['champion', 'runner_up'],
    title='Most Dominant Championship Winning Margins (Top-20)',
    labels={'gap_pts': 'Points Gap', 'season': 'Season'},
    text=top2.sort_values('gap_pts', ascending=False).head(20)['champion']
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

top2.head(20)[['season', 'champion', 'runner_up', 'champion_pts', 'runner_up_pts', 'gap_pts']]


,season,champion,runner_up,champion_pts,runner_up_pts,gap_pts
34,1984,Niki Lauda,Alain Prost,72.0,71.5,0.5
14,1964,Graham Hill,John Surtees,41.0,40.0,1.0
26,1976,James Hunt,Niki Lauda,69.0,68.0,1.0
31,1981,Nelson Piquet,Carlos Reutemann,50.0,49.0,1.0
58,2008,Lewis Hamilton,Felipe Massa,98.0,97.0,1.0
57,2007,Kimi Räikkönen,Fernando Alonso,110.0,109.0,1.0
44,1994,Michael Schumacher,Damon Hill,92.0,91.0,1.0
0,1950,Nino Farina,Luigi Fagioli,30.0,28.0,2.0
33,1983,Nelson Piquet,Alain Prost,59.0,57.0,2.0
36,1986,Alain Prost,Nigel Mansell,74.0,72.0,2.0


## 14. Constructor 1-2 Finishes

*When both cars finish first and second — the ultimate expression of team dominance.*

In [15]:

# Find races where a team scored positions 1 and 2
pos_12 = df[df['position'].isin([1, 2])].copy()
race_team_positions = pos_12.groupby(['season', 'round', 'constructor_name'])['position'].apply(set).reset_index()
race_team_positions['is_oneTwo'] = race_team_positions['position'].apply(lambda s: {1, 2}.issubset(s))

one_twos = race_team_positions[race_team_positions['is_oneTwo']]
one_two_stats = one_twos.groupby('constructor_name').size().reset_index(name='one_two_count')
one_two_stats = one_two_stats.sort_values('one_two_count', ascending=False)

# Merge with total wins for context
one_two_stats = one_two_stats.merge(const[['constructor', 'wins']], left_on='constructor_name', right_on='constructor', how='left')
one_two_stats['one_two_pct_of_wins'] = (one_two_stats['one_two_count'] / one_two_stats['wins'] * 100).round(1)

# Chart 14a — Most 1-2 finishes
fig = px.bar(
    one_two_stats.head(15).sort_values('one_two_count'),
    x='one_two_count', y='constructor_name', orientation='h',
    color='one_two_count', color_continuous_scale='Blues',
    title='Constructor 1-2 Finishes — All Time',
    labels={'one_two_count': 'Number of 1-2s', 'constructor_name': ''},
    text='one_two_count'
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()

# Season breakdown — most 1-2s in a single season
season_onetwos = one_twos.groupby(['season', 'constructor_name']).size().reset_index(name='count')
best_seasons = season_onetwos.sort_values('count', ascending=False).head(15)
best_seasons['label'] = best_seasons['constructor_name'] + ' (' + best_seasons['season'].astype(str) + ')'

fig = px.bar(
    best_seasons.sort_values('count'),
    x='count', y='label', orientation='h',
    color='count', color_continuous_scale='Oranges',
    title='Most 1-2 Finishes in a Single Season',
    labels={'count': '1-2 Finishes', 'label': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=550)
fig.show()

one_two_stats.head(15)[['constructor_name', 'one_two_count', 'wins', 'one_two_pct_of_wins']]


,constructor_name,one_two_count,wins,one_two_pct_of_wins
0,Ferrari,87,249,34.9
1,Mercedes,60,131,45.8
2,McLaren,55,199,27.6
3,Williams,33,114,28.9
4,Red Bull,31,130,23.8
5,Tyrrell,8,23,34.8
6,Team Lotus,6,45,13.3
7,Cooper-Climax,5,12,41.7
8,BRM,5,17,29.4
9,Brabham-Repco,4,8,50.0


## 15. Points Efficiency Per Season

*Which driver extracted the highest percentage of the theoretically available points in a given season? This caps the scoring at 25 pts/race × N races.*

In [16]:

# Max points per season (modern scoring: 25 pts/race max)
# We use 26 as the max per race to account for the 1 bonus fastest lap point
MAX_PTS_PER_RACE = 26

gps_per_season_map = df.groupby('season')['round'].max().to_dict()

season_pts_drv = df.groupby(['season', 'driver_fullname'])['points'].sum().reset_index()
season_pts_drv['max_available'] = season_pts_drv['season'].map(gps_per_season_map) * MAX_PTS_PER_RACE
season_pts_drv['pts_efficiency_%'] = (season_pts_drv['points'] / season_pts_drv['max_available'] * 100).round(2)

# Only modern era (2010+) for comparability
modern_eff = season_pts_drv[season_pts_drv['season'] >= 2010].sort_values('pts_efficiency_%', ascending=False)

# Chart 15a — Most efficient seasons (modern era)
top_eff = modern_eff.head(20).copy()
top_eff['label'] = top_eff['driver_fullname'] + ' (' + top_eff['season'].astype(str) + ')'

fig = px.bar(
    top_eff.sort_values('pts_efficiency_%'),
    x='pts_efficiency_%', y='label', orientation='h',
    color='pts_efficiency_%', color_continuous_scale='RdYlGn',
    title='Most Efficient Championship Seasons (2010–Present)',
    labels={'pts_efficiency_%': 'Points Efficiency %', 'label': ''},
    text=top_eff['pts_efficiency_%'].astype(str) + '%',
    hover_data=['points', 'max_available']
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 15b — Average efficiency per driver (modern era, min 3 seasons)
avg_eff = modern_eff.groupby('driver_fullname').agg(
    avg_efficiency = ('pts_efficiency_%', 'mean'),
    seasons = ('season', 'nunique'),
    total_pts = ('points', 'sum')
).reset_index()
avg_eff_filtered = avg_eff[avg_eff['seasons'] >= 3].sort_values('avg_efficiency', ascending=False)

fig = px.bar(
    avg_eff_filtered.head(20).sort_values('avg_efficiency'),
    x='avg_efficiency', y='driver_fullname', orientation='h',
    color='avg_efficiency', color_continuous_scale='Blues',
    title='Average Points Efficiency (Modern Era, Min. 3 Seasons)',
    labels={'avg_efficiency': 'Avg Efficiency %', 'driver_fullname': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

modern_eff.head(20)[['season', 'driver_fullname', 'points', 'max_available', 'pts_efficiency_%']]


,season,driver_fullname,points,max_available,pts_efficiency_%
3053,2023,Max Verstappen,530.0,572,92.66
2835,2013,Sebastian Vettel,397.0,494,80.36
2784,2011,Sebastian Vettel,392.0,494,79.35
2986,2020,Lewis Hamilton,347.0,442,78.51
2851,2014,Lewis Hamilton,384.0,494,77.73
2872,2015,Lewis Hamilton,381.0,494,77.13
3029,2022,Max Verstappen,433.0,572,75.70
2964,2019,Lewis Hamilton,413.0,546,75.64
2942,2018,Lewis Hamilton,408.0,546,74.73
2900,2016,Nico Rosberg,385.0,546,70.51


## 16. Weather Impact — Rain Races

*Wet weather transforms Formula 1. More chaos, more upsets, more legends born. Here we compare dry vs wet race statistics using a curated list of historically confirmed wet races.*

In [17]:

# Curated set of confirmed wet races (source: StatsF1, Wikipedia)
# Format: (season, round) — major rain-affected GPs from 1950-2025
WET_RACES = {
    (1968, 6), (1968, 10),  # Nurburgring, Rouen
    (1971, 7),  # Zandvoort
    (1976, 1), (1976, 11),  # Brazil, Fuji
    (1978, 14),  # Watkins Glen
    (1981, 15),  # Caesars Palace ... too many, let's use status-based heuristic instead
}

# Better approach: identify "chaotic" races via high DNF rate as a proxy for weather
# Wet races typically have much higher DNF rates and position shuffles
race_stats = df.groupby(['season', 'round', 'raceName']).agg(
    starters     = ('position', 'count'),
    finishers    = ('is_dnf', lambda x: (~x).sum()),
    dnf_count    = ('is_dnf', 'sum'),
    unique_pctg  = ('position', lambda x: (x <= 3).sum()),
    avg_pos_gain = ('positions_gained', 'mean'),
    max_pos_gain = ('positions_gained', 'max'),
    winners      = ('is_win', 'sum'),
).reset_index()
race_stats['dnf_rate'] = (race_stats['dnf_count'] / race_stats['starters'] * 100).round(1)
race_stats['chaos_index'] = (race_stats['dnf_rate'] * 0.5 + race_stats['avg_pos_gain'].abs() * 2).round(1)

# Flag top 15% most chaotic races as "likely wet/disrupted"
chaos_threshold = race_stats['chaos_index'].quantile(0.85)
race_stats['is_chaotic'] = race_stats['chaos_index'] >= chaos_threshold

# Compare chaotic vs normal races
comparison = pd.DataFrame({
    'Metric': ['Avg DNF Rate %', 'Avg Positions Gained (absolute)',
               'Max Position Gain', 'Avg Starters'],
    'Normal Races': [
        race_stats[~race_stats['is_chaotic']]['dnf_rate'].mean().round(1),
        race_stats[~race_stats['is_chaotic']]['avg_pos_gain'].abs().mean().round(2),
        race_stats[~race_stats['is_chaotic']]['max_pos_gain'].mean().round(1),
        race_stats[~race_stats['is_chaotic']]['starters'].mean().round(0),
    ],
    'Chaotic Races (Top 15%)': [
        race_stats[race_stats['is_chaotic']]['dnf_rate'].mean().round(1),
        race_stats[race_stats['is_chaotic']]['avg_pos_gain'].abs().mean().round(2),
        race_stats[race_stats['is_chaotic']]['max_pos_gain'].mean().round(1),
        race_stats[race_stats['is_chaotic']]['starters'].mean().round(0),
    ]
})
print("=== Chaotic vs Normal Race Comparison ===")
print(comparison.to_string(index=False))

# Chart 16a — Most chaotic races ever
most_chaotic = race_stats.sort_values('chaos_index', ascending=False).head(25)
most_chaotic['label'] = most_chaotic['raceName'] + ' ' + most_chaotic['season'].astype(str)

fig = px.bar(
    most_chaotic.sort_values('chaos_index'),
    x='chaos_index', y='label', orientation='h',
    color='dnf_rate', color_continuous_scale='YlOrRd',
    hover_data=['starters', 'dnf_count', 'avg_pos_gain'],
    title='Most Chaotic Races in F1 History (Chaos Index)',
    labels={'chaos_index': 'Chaos Index', 'label': '', 'dnf_rate': 'DNF Rate %'}
)
fig.update_coloraxes(colorbar_title='DNF Rate %')
fig.update_layout(height=750)
fig.show()

# Chart 16b — Chaos index by decade
race_stats['decade'] = (race_stats['season'] // 10) * 10
decade_chaos = race_stats.groupby('decade')['chaos_index'].mean().reset_index()

fig = px.bar(
    decade_chaos, x='decade', y='chaos_index',
    color='chaos_index', color_continuous_scale='Reds',
    title='Average Chaos Index by Decade — Is F1 Getting Safer?',
    labels={'decade': 'Decade', 'chaos_index': 'Avg Chaos Index'},
    text=decade_chaos['chaos_index'].round(1)
)
fig.update_traces(textposition='outside')
fig.update_coloraxes(showscale=False)
fig.update_layout(height=450)
fig.show()

# Chart 16c — Drivers who gain the most positions in chaotic races ("Rain Masters")
chaotic_race_ids = race_stats[race_stats['is_chaotic']][['season', 'round']]
chaotic_results = df.merge(chaotic_race_ids, on=['season', 'round'])
rain_masters = chaotic_results.groupby('driver_fullname').agg(
    chaotic_races = ('positions_gained', 'count'),
    avg_gain      = ('positions_gained', 'mean'),
    total_wins    = ('is_win', 'sum'),
).reset_index()
rain_masters = rain_masters[rain_masters['chaotic_races'] >= 5].sort_values('avg_gain', ascending=False)

fig = px.bar(
    rain_masters.head(20).sort_values('avg_gain'),
    x='avg_gain', y='driver_fullname', orientation='h',
    color='total_wins', color_continuous_scale='Greens',
    hover_data=['chaotic_races'],
    title='Rain Masters — Highest Avg Position Gain in Chaotic Races (Min. 5 Chaotic GPs)',
    labels={'avg_gain': 'Avg Positions Gained', 'driver_fullname': '', 'total_wins': 'Wins in Chaos'}
)
fig.update_coloraxes(colorbar_title='Wins')
fig.update_layout(height=600)
fig.show()


=== Chaotic vs Normal Race Comparison ===
                         Metric  Normal Races  Chaotic Races (Top 15%)
                 Avg DNF Rate %         31.50                    62.30
Avg Positions Gained (absolute)          0.13                     0.82
              Max Position Gain         10.70                    13.80
                   Avg Starters         22.00                    24.00


## 17. Pit Stop Strategy (2012–Present)

*Since the FIA began recording pit stop durations in 2012, the art of the pit stop has become a performance differentiator. Sub-2-second stops are now routine for the top teams.*

In [18]:

import os
pitstop_path = os.path.join('data', 'processed', 'pitstops.parquet')
if os.path.exists(pitstop_path):
    ps = pd.read_parquet(pitstop_path)
    ps = ps[ps['duration_seconds'].notna() & (ps['duration_seconds'] > 0) & (ps['duration_seconds'] < 120)]

    # Chart 17a — Average pit stop duration by season
    season_avg = ps.groupby('season')['duration_seconds'].mean().reset_index()
    fig = px.line(
        season_avg, x='season', y='duration_seconds',
        markers=True,
        title='Average Pit Stop Duration by Season (2012–Present)',
        labels={'season': 'Season', 'duration_seconds': 'Avg Duration (seconds)'}
    )
    fig.update_traces(line_color='#FF1801', line_width=3)
    fig.update_layout(height=400)
    fig.show()

    # Chart 17b — Fastest pit stops ever
    fastest = ps.sort_values('duration_seconds').head(20).copy()
    fastest['label'] = fastest['driver_fullname'].fillna(fastest['driverId']) + ' (' + fastest['season'].astype(str) + ' ' + fastest['raceName'] + ')'
    fig = px.bar(
        fastest.sort_values('duration_seconds', ascending=False),
        x='duration_seconds', y='label', orientation='h',
        color='duration_seconds', color_continuous_scale='RdYlGn_r',
        title='20 Fastest Pit Stops in F1 History (2012+)',
        labels={'duration_seconds': 'Duration (s)', 'label': ''},
        text=fastest['duration_seconds'].round(2)
    )
    fig.update_traces(textposition='outside')
    fig.update_coloraxes(showscale=False)
    fig.update_layout(height=600)
    fig.show()

    # Chart 17c — Average stop time by constructor (all time)
    constructor_map = df[['driverId', 'constructor_name']].drop_duplicates()
    ps_with_team = ps.merge(constructor_map, on='driverId', how='left')
    team_avg = ps_with_team.groupby('constructor_name').agg(
        avg_duration = ('duration_seconds', 'mean'),
        total_stops  = ('duration_seconds', 'count'),
    ).reset_index()
    team_avg = team_avg[team_avg['total_stops'] >= 50].sort_values('avg_duration')

    fig = px.bar(
        team_avg.head(20),
        x='avg_duration', y='constructor_name', orientation='h',
        color='avg_duration', color_continuous_scale='RdYlGn_r',
        title='Fastest Pit Stop Teams (Avg Duration, Min. 50 Stops)',
        labels={'avg_duration': 'Avg Duration (s)', 'constructor_name': ''},
        text=team_avg.head(20)['avg_duration'].round(2)
    )
    fig.update_traces(textposition='outside')
    fig.update_coloraxes(showscale=False)
    fig.update_layout(height=600)
    fig.show()

    # Chart 17d — Number of stops per race distribution
    stops_per_race = ps.groupby(['season', 'round', 'driverId'])['stop_number'].max().reset_index()
    stop_dist = stops_per_race['stop_number'].value_counts().sort_index().reset_index()
    stop_dist.columns = ['num_stops', 'count']

    fig = px.bar(
        stop_dist, x='num_stops', y='count',
        color='count', color_continuous_scale='Blues',
        title='Pit Stop Strategy Distribution (Number of Stops Per Driver Per Race)',
        labels={'num_stops': 'Number of Pit Stops', 'count': 'Occurrences'}
    )
    fig.update_coloraxes(showscale=False)
    fig.update_layout(height=400)
    fig.show()

    print(f"Total pit stops analyzed: {len(ps)}")
    print(f"Fastest ever: {ps['duration_seconds'].min():.2f}s")
    print(f"Average: {ps['duration_seconds'].mean():.2f}s")
else:
    print("Pit stop data not available. Run: python src/data/ingest_pitstops.py")


Total pit stops analyzed: 10587
Fastest ever: 12.80s
Average: 24.64s


## 18. Safety & Reliability Evolution

*From the deadly early decades to modern safety standards, how has F1's reliability and danger evolved? DNF causes tell the story of engineering progress.*

In [19]:

# DNF causes classified by era
df['era'] = pd.cut(df['season'],
    bins=[1949, 1969, 1979, 1989, 1999, 2009, 2025],
    labels=['1950s-60s', '1970s', '1980s', '1990s', '2000s', '2010s-20s']
)

# Classify DNF causes
MECHANICAL = ['Engine', 'Gearbox', 'Transmission', 'Clutch', 'Hydraulics',
              'Electrical', 'Suspension', 'Brakes', 'Overheating', 'Oil pressure',
              'Fuel pressure', 'Water pressure', 'Oil leak', 'Fuel leak',
              'Throttle', 'Steering', 'Exhaust', 'Turbo', 'Differential']
ACCIDENT = ['Accident', 'Collision', 'Spun off', 'Collision damage']

def classify_dnf(status):
    if status in MECHANICAL:
        return 'Mechanical'
    elif status in ACCIDENT:
        return 'Accident/Collision'
    elif status in ['Retired', 'Withdrew', 'Not classified', 'Disqualified']:
        return 'Other'
    else:
        return 'Other'

dnf_data = df[df['is_dnf']].copy()
dnf_data['dnf_category'] = dnf_data['status'].apply(classify_dnf)

# Chart 18a — DNF cause breakdown by era
era_dnf = dnf_data.groupby(['era', 'dnf_category']).size().reset_index(name='count')

fig = px.bar(
    era_dnf, x='era', y='count', color='dnf_category',
    barmode='group',
    color_discrete_map={'Mechanical': '#FF6B35', 'Accident/Collision': '#D32F2F', 'Other': '#757575'},
    title='DNF Causes by Era — Mechanical vs Accident',
    labels={'era': 'Era', 'count': 'Number of DNFs', 'dnf_category': 'Cause'}
)
fig.update_layout(height=500)
fig.show()

# Chart 18b — DNF rate evolution by season
season_dnf = df.groupby('season').agg(
    total   = ('is_dnf', 'count'),
    dnf_sum = ('is_dnf', 'sum'),
).reset_index()
season_dnf['dnf_rate'] = (season_dnf['dnf_sum'] / season_dnf['total'] * 100).round(1)

fig = px.area(
    season_dnf, x='season', y='dnf_rate',
    title='DNF Rate Evolution (1950–Present) — The Safety Revolution',
    labels={'season': 'Season', 'dnf_rate': 'DNF Rate %'}
)
fig.update_traces(fill='tozeroy', line_color='#D32F2F')
fig.update_layout(height=400)
fig.show()

# Chart 18c — Most reliable drivers (lowest DNF rate, min 50 starts)
drv_reliability = df.groupby('driver_fullname').agg(
    starts = ('is_dnf', 'count'),
    dnfs   = ('is_dnf', 'sum'),
).reset_index()
drv_reliability['dnf_rate'] = (drv_reliability['dnfs'] / drv_reliability['starts'] * 100).round(1)
drv_reliable = drv_reliability[drv_reliability['starts'] >= 50].sort_values('dnf_rate')

fig = px.bar(
    drv_reliable.head(20).sort_values('dnf_rate', ascending=False),
    x='dnf_rate', y='driver_fullname', orientation='h',
    color='dnf_rate', color_continuous_scale='RdYlGn_r',
    title='Most Reliable Drivers (Lowest DNF Rate, Min. 50 Starts)',
    labels={'dnf_rate': 'DNF Rate %', 'driver_fullname': ''},
    hover_data=['starts', 'dnfs']
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Table
season_dnf.sort_values('dnf_rate').head(10)[['season', 'total', 'dnf_sum', 'dnf_rate']]


,season,total,dnf_sum,dnf_rate
74,2024,479,47,9.8
73,2023,440,50,11.4
75,2025,479,55,11.5
63,2013,418,49,11.7
71,2021,440,52,11.8
69,2019,420,50,11.9
72,2022,440,64,14.5
70,2020,340,53,15.6
62,2012,478,76,15.9
66,2016,462,79,17.1


## 19. Elo Rating — Comparing Drivers Across Eras

*An Elo rating system adapted for F1: each Grand Prix generates head-to-head matchups between all finishers. K-factor=6, initial Elo=1500. This allows cross-era comparison on a single scale.*

In [20]:

# --- Elo Rating computation ---
K_FACTOR = 6
INITIAL_ELO = 1500

# Sort races chronologically
races_chrono = df[df['position'].notna() & (df['position'] > 0)].sort_values(['date', 'season', 'round'])
race_keys = races_chrono.groupby(['season', 'round']).first().reset_index()[['season', 'round', 'date']].values

elo_ratings = {}  # driverId -> current Elo
elo_history = []  # list of (date, season, round, driverId, elo)

def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

for season, rnd, date in race_keys:
    race_results = races_chrono[
        (races_chrono['season'] == season) & (races_chrono['round'] == rnd)
    ][['driverId', 'driver_fullname', 'position']].copy()

    drivers_in_race = race_results['driverId'].tolist()

    # Initialize new drivers
    for did in drivers_in_race:
        if did not in elo_ratings:
            elo_ratings[did] = INITIAL_ELO

    # Head-to-head matchups: every pair of finishers
    for i, row_a in race_results.iterrows():
        for j, row_b in race_results.iterrows():
            if row_a['driverId'] >= row_b['driverId']:
                continue
            da, db = row_a['driverId'], row_b['driverId']
            ra, rb = elo_ratings[da], elo_ratings[db]
            ea, eb = expected_score(ra, rb), expected_score(rb, ra)
            # Actual score: 1 if beat the other, 0 if lost
            sa = 1.0 if row_a['position'] < row_b['position'] else 0.0
            sb = 1.0 - sa
            elo_ratings[da] += K_FACTOR * (sa - ea)
            elo_ratings[db] += K_FACTOR * (sb - eb)

    # Record snapshot after race
    for did in drivers_in_race:
        name = race_results[race_results['driverId'] == did]['driver_fullname'].iloc[0]
        elo_history.append((date, season, rnd, did, name, round(elo_ratings[did], 1)))

elo_df = pd.DataFrame(elo_history, columns=['date', 'season', 'round', 'driverId', 'driver', 'elo'])

# Peak Elo per driver
peak_elo = elo_df.groupby(['driverId', 'driver']).agg(
    peak_elo = ('elo', 'max'),
    final_elo = ('elo', 'last'),
    races = ('elo', 'count'),
).reset_index().sort_values('peak_elo', ascending=False)

# Chart 19a — Highest Peak Elo All-Time
fig = px.bar(
    peak_elo.head(25).sort_values('peak_elo'),
    x='peak_elo', y='driver', orientation='h',
    color='peak_elo', color_continuous_scale='Turbo',
    title='All-Time Highest Elo Rating in F1 History',
    labels={'peak_elo': 'Peak Elo Rating', 'driver': ''},
    hover_data=['final_elo', 'races']
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=750)
fig.show()

# Chart 19b — Elo evolution of the top 10 all-time drivers
top10_ids = peak_elo.head(10)['driverId'].tolist()
top10_elo = elo_df[elo_df['driverId'].isin(top10_ids)]

fig = px.line(
    top10_elo, x='date', y='elo', color='driver',
    title='Elo Rating Evolution — Top 10 All-Time Drivers',
    labels={'date': 'Date', 'elo': 'Elo Rating', 'driver': 'Driver'}
)
fig.update_layout(height=600)
fig.show()

# Table — Top 25 with peak and final Elo
peak_elo.head(25)[['driver', 'peak_elo', 'final_elo', 'races']]


,driver,peak_elo,final_elo,races
493,Max Verstappen,2231.6,2088.7,233
326,Lewis Hamilton,2193.8,1740.4,380
772,Sebastian Vettel,2101.9,1643.5,300
655,Nico Rosberg,2094.9,2094.9,206
589,Oscar Piastri,2092.2,1933.9,70
552,Lando Norris,2077.1,1959.2,152
511,Michael Schumacher,2059.6,1603.4,308
99,Valtteri Bottas,2052.2,1438.7,247
19,Fernando Alonso,2044.1,1669.1,428
579,Sergio Pérez,1998.4,1598.9,283
